# OmniVoice Project Studio — Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/feat/section-status-resume/notebooks/OmniVoice_Project_Studio_Colab.ipynb)

Long-form workflow with Text Doctor, persistent section resume, live status, and adaptive quality control:

**Text Doctor → Voice Library → Project → section-status.json → Preflight → Generate incomplete sections → Adaptive Retry + Pacing Guard → Resume → Preview → Merge**

Text Doctor only applies safe formatting/encoding fixes automatically. Numbers, abbreviations, and unknown directives are shown as review hints instead of being silently rewritten.


In [ ]:
# Install the persistent section-resume Project Studio branch
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@feat/section-status-resume"

import torch
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime: Runtime → Change runtime type → T4 GPU")


## Mount Google Drive

Projects, encoded voice prompts, generated WAV files, section checkpoints, adaptive verification reports, and diagnostic reports persist under:

`MyDrive/OmniVoiceStudio/`

Each project now contains an independent `section-status.json`. A section is considered complete only when its status is `verified` and its final section WAV still exists. Interrupted `queued`/`generating` states are recovered as pending on the next load.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/OmniVoiceStudio"


## Launch Project Studio

The first launch downloads OmniVoice model weights and the Whisper verifier.

Recommended flow:

1. Open **1. Text Doctor**, paste the full Markdown script, and click **Analyze & clean safe issues**.
2. Review the visible diff and review hints. Copy the cleaned script.
3. Open **2. Project Studio → Project Setup / Legacy**, save/select a voice and create the project from the cleaned script.
4. Open **Generate / Resume**, select project + voice + `AUTO`, then click **Preflight**.
5. Click **Generate / Resume**. Every Sxx row remains visible and updates section by section.
6. If Colab disconnects, reopen this notebook and load the same project. The resume controller reads `section-status.json`, skips verified sections whose WAV exists, and continues only incomplete sections.

Adaptive quality checks each chunk for missing/repeated text and pacing anomalies. Per-chunk JSON reports include `recovered_from`, `retry_actions`, `global_wps`, `max_local_wps`, and `pacing_anomaly`.


In [ ]:
!python -m omnivoice.cli.project_studio_text_resume \
  --model k2-fsa/OmniVoice \
  --workspace "/content/drive/MyDrive/OmniVoiceStudio" \
  --asr-model openai/whisper-small.en \
  --asr-device cpu \
  --share
